# Boids Flocking Simulation

A Python-driven [boids](https://en.wikipedia.org/wiki/Boids) flocking simulation powered by the Minkowski ECS engine.
Each boid follows three simple rules — **separation** (steer away from nearby neighbors),
**alignment** (match heading with nearby neighbors), and **cohesion** (steer toward the
center of nearby neighbors). These local rules produce emergent flocking behavior.

All heavy lifting (neighbor queries, force accumulation, integration) runs in compiled Rust
reducers exposed to Python through the `minkowski_py` bridge. Python handles orchestration,
parameter tuning, and visualization.

In [ ]:
import random

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import minkowski_py as mk
from IPython.display import clear_output, display

# Constants
N_BOIDS = 1000
WORLD_SIZE = 500.0
FRAMES = 200
DT = 1.0

## Spawn Boids

Each boid starts at a random position in `[0, WORLD_SIZE)` with a random velocity in `[-2, 2]`.
Acceleration is initialized to zero — the flocking reducers will populate it each frame.

In [ ]:
random.seed(42)

world = mk.World()

# Build spawn data — keys must be alphabetically sorted
spawn_data = {
    "acc_x": [0.0] * N_BOIDS,
    "acc_y": [0.0] * N_BOIDS,
    "pos_x": [random.uniform(0, WORLD_SIZE) for _ in range(N_BOIDS)],
    "pos_y": [random.uniform(0, WORLD_SIZE) for _ in range(N_BOIDS)],
    "vel_x": [random.uniform(-2, 2) for _ in range(N_BOIDS)],
    "vel_y": [random.uniform(-2, 2) for _ in range(N_BOIDS)],
}

world.spawn_batch("Acceleration,Position,Velocity", spawn_data)
print(f"Spawned {N_BOIDS} boids")

In [ ]:
registry = mk.ReducerRegistry(world)

## Simulation Parameters

The flocking behavior is controlled by six parameters:

| Parameter | Description |
|-----------|-------------|
| `sep_r` / `sep_w` | Separation radius and weight — steer away from boids within this range |
| `ali_r` / `ali_w` | Alignment radius and weight — match velocity of boids within this range |
| `coh_r` / `coh_w` | Cohesion radius and weight — steer toward center of mass within this range |
| `max_force` | Clamps the total steering force per frame |
| `max_speed` | Clamps velocity magnitude after integration |

In [ ]:
boid_params = dict(
    world_size=WORLD_SIZE,
    sep_r=25.0,
    ali_r=50.0,
    coh_r=50.0,
    sep_w=1.5,
    ali_w=1.0,
    coh_w=1.0,
    max_force=1.0,
)

integrate_params = dict(
    max_speed=4.0,
    dt=DT,
    world_size=WORLD_SIZE,
)

## Run Simulation

Each frame calls two Rust reducers:
1. **`boids_forces`** — computes separation/alignment/cohesion forces and writes them to `Acceleration`
2. **`boids_integrate`** — updates `Velocity` from `Acceleration`, then updates `Position` from `Velocity` (toroidal wrap)

We snapshot positions and velocities every 10 frames for animation.

In [ ]:
snapshots = []  # list of (pos_x, pos_y, vel_x, vel_y) arrays

for frame in range(FRAMES):
    registry.run("boids_forces", world, **boid_params)
    registry.run("boids_integrate", world, **integrate_params)

    if frame % 10 == 0:
        df = world.query("Position", "Velocity")
        snapshots.append(
            (
                df["pos_x"].to_numpy(),
                df["pos_y"].to_numpy(),
                df["vel_x"].to_numpy(),
                df["vel_y"].to_numpy(),
            )
        )

    if frame % 50 == 0:
        print(f"Frame {frame}/{FRAMES}")

print(f"Done — {len(snapshots)} snapshots captured")

## Visualize Final Frame

Scatter plot of all boids colored by speed (velocity magnitude). Faster boids appear warmer.

In [ ]:
import numpy as np

pos_x, pos_y, vel_x, vel_y = snapshots[-1]
speed = np.sqrt(vel_x**2 + vel_y**2)

fig, ax = plt.subplots(figsize=(8, 8))
sc = ax.scatter(pos_x, pos_y, c=speed, cmap="plasma", s=4, alpha=0.8)
ax.set_xlim(0, WORLD_SIZE)
ax.set_ylim(0, WORLD_SIZE)
ax.set_aspect("equal")
ax.set_title(f"Boids — Frame {FRAMES}")
ax.set_xlabel("x")
ax.set_ylabel("y")
fig.colorbar(sc, ax=ax, label="Speed")
plt.tight_layout()
plt.show()

## Animation

Step through the captured snapshots. Uses `clear_output` for a simple frame-by-frame animation.

> For smoother animation in JupyterLab, consider using `ipywidgets` or `matplotlib.animation.FuncAnimation`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(0, WORLD_SIZE)
ax.set_ylim(0, WORLD_SIZE)
ax.set_aspect("equal")
ax.set_xlabel("x")
ax.set_ylabel("y")

# Initial scatter
pos_x, pos_y, vel_x, vel_y = snapshots[0]
speed = np.sqrt(vel_x**2 + vel_y**2)
sc = ax.scatter(pos_x, pos_y, c=speed, cmap="plasma", s=4, alpha=0.8, vmin=0, vmax=5)
title = ax.set_title("Boids — Frame 0")

for i, (px, py, vx, vy) in enumerate(snapshots):
    spd = np.sqrt(vx**2 + vy**2)
    sc.set_offsets(np.column_stack([px, py]))
    sc.set_array(spd)
    title.set_text(f"Boids — Frame {i * 10}")
    clear_output(wait=True)
    display(fig)

plt.close(fig)
print("Animation complete")

## Spatial Analysis

Demonstrate the `SpatialGrid` API. We rebuild the grid from the final world state,
pick entity 0, and highlight its neighbors within the separation radius.

In [ ]:
SEP_R = boid_params["sep_r"]

# Build spatial grid from current world state
grid = mk.SpatialGrid(world_size=WORLD_SIZE, cell_size=SEP_R)
grid.rebuild(world)

# Query final positions
df = world.query("Position", "Velocity")
all_pos_x = df["pos_x"].to_numpy()
all_pos_y = df["pos_y"].to_numpy()
entity_ids = df["entity_id"].to_numpy()

# Pick entity 0 and find its neighbors
target_idx = 0
target_eid = entity_ids[target_idx]
tx, ty = all_pos_x[target_idx], all_pos_y[target_idx]
neighbor_eids = set(grid.query_radius(tx, ty, SEP_R))
neighbor_eids.discard(target_eid)

# Map entity IDs to indices for coloring
eid_to_idx = {eid: i for i, eid in enumerate(entity_ids)}
neighbor_idxs = [eid_to_idx[eid] for eid in neighbor_eids if eid in eid_to_idx]

# Color assignment: blue=default, orange=neighbor, red=target
colors = np.full(len(entity_ids), 0)  # 0=blue
colors[neighbor_idxs] = 1  # 1=orange
colors[target_idx] = 2  # 2=red
cmap = mcolors.ListedColormap(["steelblue", "orange", "red"])
sizes = np.full(len(entity_ids), 4.0)
sizes[neighbor_idxs] = 20.0
sizes[target_idx] = 60.0

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(
    all_pos_x, all_pos_y, c=colors, cmap=cmap, s=sizes, alpha=0.8, vmin=0, vmax=2
)

# Draw radius circle around target
circle = plt.Circle(
    (tx, ty), SEP_R, fill=False, color="red", linewidth=1.5, linestyle="--"
)
ax.add_patch(circle)

ax.set_xlim(0, WORLD_SIZE)
ax.set_ylim(0, WORLD_SIZE)
ax.set_aspect("equal")
ax.set_title(f"Neighbors of Entity 0 (r={SEP_R:.0f}) — {len(neighbor_eids)} found")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.tight_layout()
plt.show()

## Explore

Try tweaking the simulation parameters and re-running:

- **More spacing**: increase `sep_w` (e.g., 3.0) — boids repel harder, producing looser formations
- **Tighter flocks**: increase `coh_w` (e.g., 2.0) — boids pull toward their local center of mass
- **Faster/slower**: adjust `max_speed` and `max_force` — higher values give more dynamic motion
- **Fewer boids**: reduce `N_BOIDS` for faster iteration while experimenting
- **Larger neighborhoods**: increase `ali_r` or `coh_r` — boids coordinate over longer distances